In [3]:
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
import time
import zipfile
from pathlib import Path
from IPython.display import display


# =============================================================================
# 07_SetStop_External_Baselines_NSM_ReaRev.ipynb
# One-cell evaluation notebook. Run on a T4 GPU.
# =============================================================================

ROOT = Path("/content/drive/MyDrive/SetStop_KGQA")
DATA_DIR = ROOT / "data/extracted/webqsp"
SETSTOP_SUMMARY = (
    ROOT / "outputs/end_to_end_test_v1/end_to_end_summary_v1.json"
)

OUT_DIR = ROOT / "outputs/external_baselines_v1"
CACHE_DIR = ROOT / "external_baselines/cache"

WORK_DIR = Path("/content/setstop_external_baselines_v1")
COMPAT_DATA = WORK_DIR / "data"

NSM_REPO = WORK_DIR / "WSDM2021_NSM"
REAREV_REPO = WORK_DIR / "ReaRev_KGQA"

NSM_CKPT_DIR = WORK_DIR / "nsm_checkpoint"
REAREV_CKPT_DIR = WORK_DIR / "rearev_checkpoint"

REAREV_LOG = OUT_DIR / "rearev_execution_log_v1.txt"

for folder in [
    OUT_DIR,
    CACHE_DIR,
    WORK_DIR,
    COMPAT_DATA,
    NSM_CKPT_DIR,
    REAREV_CKPT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

print("=" * 100)
print("SETSTOP-KGQA CONTROLLED EXTERNAL BASELINES: NSM+h AND ReaRev")
print("=" * 100)


# =============================================================================
# Helper functions
# =============================================================================

def run(command, cwd=None, env=None):
    command = [str(x) for x in command]

    print("\nRUN:", " ".join(command))

    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if completed.stdout:
        print(completed.stdout)

    if completed.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {completed.returncode}\n"
            f"Command: {' '.join(command)}\n\n"
            f"Output:\n{completed.stdout}"
        )

    return completed


def run_logged(command, cwd, log_path, env=None):
    """
    Run a process while:
    1. showing output in Colab,
    2. saving the full output to Drive,
    3. preserving the final traceback if the command fails.
    """

    command = [str(x) for x in command]

    print("\nRUN:", " ".join(command))

    merged_env = os.environ.copy()

    if env:
        merged_env.update(
            {str(key): str(value) for key, value in env.items()}
        )

    local_log = WORK_DIR / f"{Path(log_path).name}.temporary"

    recent_lines = []

    with open(local_log, "w", encoding="utf-8") as log_handle:

        process = subprocess.Popen(
            command,
            cwd=str(cwd),
            env=merged_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        if process.stdout is None:
            raise RuntimeError("Could not capture subprocess output.")

        for line in process.stdout:
            print(line, end="")
            log_handle.write(line)
            log_handle.flush()

            recent_lines.append(line.rstrip("\n"))

            if len(recent_lines) > 200:
                recent_lines.pop(0)

        return_code = process.wait()

    shutil.copy2(local_log, log_path)

    if return_code != 0:
        final_output = "\n".join(recent_lines[-100:])

        raise RuntimeError(
            f"ReaRev failed with exit code {return_code}.\n"
            f"Complete log saved to:\n{log_path}\n\n"
            f"Last subprocess output:\n{final_output}"
        )


def sha256_file(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)

    return digest.hexdigest()


def download_gdrive(file_id, destination, expected_sha256):
    destination = Path(destination)

    if destination.exists():
        existing_hash = sha256_file(destination)

        if existing_hash != expected_sha256:
            print("Removing invalid cached file:", destination)
            destination.unlink()

    if not destination.exists():
        run([
            sys.executable,
            "-m",
            "gdown",
            file_id,
            "-O",
            destination,
        ])

    actual_hash = sha256_file(destination)

    if actual_hash != expected_sha256:
        raise RuntimeError(
            f"Checksum mismatch for {destination.name}\n"
            f"Expected: {expected_sha256}\n"
            f"Found:    {actual_hash}"
        )

    print("Verified:", destination.name, actual_hash[:16])


def ensure_repo(url, destination, commit, required_file):
    destination = Path(destination)

    if not (destination / required_file).exists():

        if destination.exists():
            shutil.rmtree(destination)

        run([
            "git",
            "clone",
            "--depth",
            "1",
            url,
            destination,
        ])

    current_commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(destination),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()

    if current_commit != commit:

        run([
            "git",
            "-C",
            destination,
            "fetch",
            "--depth",
            "1",
            "origin",
            commit,
        ])

        run([
            "git",
            "-C",
            destination,
            "checkout",
            "--detach",
            commit,
        ])

        current_commit = subprocess.check_output(
            [
                "git",
                "-C",
                str(destination),
                "rev-parse",
                "HEAD",
            ],
            text=True,
        ).strip()

    if current_commit != commit:
        raise RuntimeError(
            f"Repository commit mismatch for {destination.name}\n"
            f"Expected: {commit}\n"
            f"Found:    {current_commit}"
        )

    print(destination.name, "commit:", current_commit)

    return current_commit


def replace_if_present(path, old, new):
    path = Path(path)

    source = path.read_text(encoding="utf-8")

    if old in source:
        source = source.replace(old, new)
        path.write_text(source, encoding="utf-8")


def valid_raw_file(path, expected_rows=854):
    path = Path(path)

    if not path.exists():
        return False

    try:
        with open(path, encoding="utf-8") as handle:
            rows = [
                line
                for line in handle
                if line.strip()
            ]

        if len(rows) != expected_rows:
            return False

        # Also verify that every line is valid JSON.
        for line in rows:
            json.loads(line)

        return True

    except Exception:
        return False


# =============================================================================
# Install required packages
# =============================================================================

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "gdown>=5.2.0",
    "pandas>=2.0",
    "tqdm>=4.60",
])

import numpy as np
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Colab, select:\n"
        "Runtime > Change runtime type > T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
)

try:
    transformers_before = importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError:
    transformers_before = "not installed"

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Transformers before compatibility install:", transformers_before)
print("GPU:", gpu_name, f"({gpu_memory_gb:.2f} GB)")


# =============================================================================
# Install ReaRev-compatible Transformers
#
# ReaRev originally used Transformers 4.6.1. That exact release cannot be
# installed reliably on Python 3.13 because its old tokenizers package does
# not provide a compatible wheel.
#
# Transformers 4.46.3 keeps the required Transformers 4 API while supporting
# modern Python and PyTorch.
#
# ReaRev runs as a fresh Python subprocess, so no runtime restart is required.
# =============================================================================

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--upgrade",
    "transformers==4.46.3",
    "tokenizers==0.20.3",
    "huggingface-hub==0.26.5",
    "safetensors==0.4.5",
])

compatibility_check = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import torch, transformers; "
            "print('Transformers:', transformers.__version__); "
            "print('PyTorch:', torch.__version__); "
            "print('CUDA:', torch.cuda.is_available()); "
            "assert transformers.__version__ == '4.46.3'; "
            "assert torch.cuda.is_available()"
        ),
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(compatibility_check.stdout)

if compatibility_check.returncode != 0:
    raise RuntimeError(
        "The Transformers 4 compatibility installation failed.\n\n"
        + compatibility_check.stdout
    )

rearev_transformers_version = subprocess.check_output(
    [
        sys.executable,
        "-c",
        "import transformers; print(transformers.__version__)",
    ],
    text=True,
).strip()

print(
    "ReaRev subprocess Transformers:",
    rearev_transformers_version,
)


# =============================================================================
# Verify WebQSP data
# =============================================================================

required_data_files = [
    "train_simple.json",
    "dev_simple.json",
    "test_simple.json",
    "train.dep",
    "dev.dep",
    "test.dep",
    "entities.txt",
    "relations.txt",
    "vocab_new.txt",
    "word_emb_300d.npy",
]

for filename in required_data_files:
    path = DATA_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required dataset file:\n{path}"
        )

print("WebQSP dataset verification passed.")


# =============================================================================
# Clone official repositories
# =============================================================================

NSM_COMMIT = ensure_repo(
    url="https://github.com/RichardHGL/WSDM2021_NSM.git",
    destination=NSM_REPO,
    commit="2e20915956a69b6f832143ded9d54442ccbe698e",
    required_file="main_nsm.py",
)

REAREV_COMMIT = ensure_repo(
    url="https://github.com/cmavro/ReaRev_KGQA.git",
    destination=REAREV_REPO,
    commit="7d2a7786870c6b7a20d7c295bff7b8790cf41c03",
    required_file="main.py",
)


# =============================================================================
# Download official checkpoints
# =============================================================================

NSM_ARCHIVE = CACHE_DIR / "nsm_report_ckpt.tgz"
REAREV_ARCHIVE = CACHE_DIR / "rearev_checkpoint.zip"

download_gdrive(
    file_id="15J02zSJTZUFyeBv-hk-2FII3qEoIVyr2",
    destination=NSM_ARCHIVE,
    expected_sha256=(
        "61ff8d157645ebe20048c0a4f77ebbc6"
        "4150e0149c7c6cefb4650d0d3d9399d2"
    ),
)

download_gdrive(
    file_id="1p7eLSsSKkZQxB32mT5lMsthVP6R_3x1j",
    destination=REAREV_ARCHIVE,
    expected_sha256=(
        "239fcff6bc3a0dffea72d24e3fc53830"
        "ea92fb56cce2160ffbafb43b72328227"
    ),
)

NSM_CKPT = NSM_CKPT_DIR / "webqsp_nsm_h_final.ckpt"
REAREV_CKPT = REAREV_CKPT_DIR / "ReaRev_webqsp.ckpt"

if not NSM_CKPT.exists():

    archive_member = (
        "webqsp_report/student/"
        "webqsp_t_gnn_s_gnn_fb_js-final.ckpt"
    )

    with tarfile.open(NSM_ARCHIVE, "r:gz") as archive:
        source = archive.extractfile(archive_member)

        if source is None:
            raise FileNotFoundError(
                f"Checkpoint not found in archive: {archive_member}"
            )

        with source, open(NSM_CKPT, "wb") as target:
            shutil.copyfileobj(source, target)

if not REAREV_CKPT.exists():

    archive_member = "checkpoint/pretrain/ReaRev_webqsp.ckpt"

    with zipfile.ZipFile(REAREV_ARCHIVE) as archive:
        with archive.open(archive_member) as source:
            with open(REAREV_CKPT, "wb") as target:
                shutil.copyfileobj(source, target)

expected_nsm_checkpoint_hash = (
    "9a2ed37a0be28ad12bc7f3af69d9850c"
    "c360a3deda6f44f7babe388f631b06c8"
)

expected_rearev_checkpoint_hash = (
    "e6bd55d85ea06a60b89569e94e26b2e"
    "474a8d5bfc9b7d9b6cd6d12ea5a699687"
)

actual_nsm_checkpoint_hash = sha256_file(NSM_CKPT)
actual_rearev_checkpoint_hash = sha256_file(REAREV_CKPT)

if actual_nsm_checkpoint_hash != expected_nsm_checkpoint_hash:
    raise RuntimeError(
        "Extracted NSM checkpoint checksum mismatch."
    )

if actual_rearev_checkpoint_hash != expected_rearev_checkpoint_hash:
    raise RuntimeError(
        "Extracted ReaRev checkpoint checksum mismatch."
    )

print("Checkpoint extraction and verification passed.")


# =============================================================================
# Compatibility patches
# =============================================================================

replace_if_present(
    NSM_REPO / "NSM/train/trainer_nsm.py",
    "checkpoint = torch.load(filename)",
    (
        "checkpoint = torch.load("
        "filename, map_location=self.device, weights_only=False)"
    ),
)

replace_if_present(
    REAREV_REPO / "train_model.py",
    "checkpoint = torch.load(filename)",
    (
        "checkpoint = torch.load("
        "filename, map_location=self.device, weights_only=False)"
    ),
)

replace_if_present(
    REAREV_REPO / "dataset_load.py",
    "pad_to_max_length=True",
    "padding='max_length'",
)

print("Repository compatibility patches applied.")


# =============================================================================
# Recreate the controlled multi-answer subset
#
# Inclusion rule:
# - between 2 and 50 mapped gold answers;
# - every mapped gold answer is present in the question subgraph.
# =============================================================================

entity_ids = [
    line.strip()
    for line in open(
        DATA_DIR / "entities.txt",
        encoding="utf-8",
    )
]

entity_to_index = {
    entity_id: index
    for index, entity_id in enumerate(entity_ids)
}


def graph_entity_index(value):

    if isinstance(value, int):
        return value

    if isinstance(value, str):
        return entity_to_index.get(value)

    if isinstance(value, dict):
        raw_value = value.get(
            "text",
            value.get("kb_id"),
        )

        if isinstance(raw_value, int):
            return raw_value

        return entity_to_index.get(raw_value)

    return None


def gold_entity_ids(row):

    result = set()

    for answer in row["answers"]:

        kb_id = answer.get("kb_id")
        answer_text = answer.get("text")

        if isinstance(kb_id, str) and kb_id in entity_to_index:
            result.add(kb_id)

        elif (
            isinstance(kb_id, int)
            and 0 <= kb_id < len(entity_ids)
        ):
            result.add(entity_ids[kb_id])

        elif (
            isinstance(answer_text, str)
            and answer_text in entity_to_index
        ):
            result.add(answer_text)

    return result


def load_selected_split(split):

    simple_path = DATA_DIR / f"{split}_simple.json"
    dependency_path = DATA_DIR / f"{split}.dep"

    simple_lines = simple_path.read_text(
        encoding="utf-8"
    ).splitlines()

    dependency_lines = dependency_path.read_text(
        encoding="utf-8"
    ).splitlines()

    if len(simple_lines) != len(dependency_lines):
        raise AssertionError(
            f"{split}: JSON/dependency length mismatch."
        )

    selected_rows = []
    selected_dependencies = []

    for simple_line, dependency_line in zip(
        simple_lines,
        dependency_lines,
    ):
        row = json.loads(simple_line)

        gold_ids = gold_entity_ids(row)

        gold_indices = {
            entity_to_index[entity_id]
            for entity_id in gold_ids
        }

        graph_indices = {
            index
            for index in (
                graph_entity_index(value)
                for value in row["subgraph"]["entities"]
            )
            if index is not None
        }

        if (
            2 <= len(gold_indices) <= 50
            and gold_indices.issubset(graph_indices)
        ):
            selected_rows.append(row)
            selected_dependencies.append(dependency_line)

    return selected_rows, selected_dependencies


dev_rows, dev_dependencies = load_selected_split("dev")
test_rows, test_dependencies = load_selected_split("test")

if len(dev_rows) != 118:
    raise AssertionError(
        f"Expected 118 development questions, found {len(dev_rows)}"
    )

if len(test_rows) != 736:
    raise AssertionError(
        f"Expected 736 test questions, found {len(test_rows)}"
    )

dev_qids = {
    str(row["id"])
    for row in dev_rows
}

test_qids = {
    str(row["id"])
    for row in test_rows
}

if not dev_qids.isdisjoint(test_qids):
    raise AssertionError(
        "Development and test question IDs overlap."
    )

joint_rows = dev_rows + test_rows
joint_dependencies = (
    dev_dependencies + test_dependencies
)

if len(joint_rows) != 854:
    raise AssertionError(
        f"Expected 854 joint questions, found {len(joint_rows)}"
    )

if len({str(row["id"]) for row in joint_rows}) != 854:
    raise AssertionError(
        "Duplicate question IDs found in controlled subset."
    )


def write_jsonl(path, rows):

    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


write_jsonl(
    COMPAT_DATA / "test_simple.json",
    joint_rows,
)

(COMPAT_DATA / "test.dep").write_text(
    "\n".join(joint_dependencies) + "\n",
    encoding="utf-8",
)

write_jsonl(
    COMPAT_DATA / "dev.json",
    dev_rows,
)

write_jsonl(
    COMPAT_DATA / "test.json",
    joint_rows,
)


def safe_symlink(source, destination):

    source = Path(source)
    destination = Path(destination)

    if destination.exists() or destination.is_symlink():
        destination.unlink()

    destination.symlink_to(source)


for filename in [
    "entities.txt",
    "relations.txt",
    "vocab_new.txt",
    "word_emb_300d.npy",
]:
    safe_symlink(
        DATA_DIR / filename,
        COMPAT_DATA / filename,
    )

safe_symlink(
    DATA_DIR / "vocab_new.txt",
    COMPAT_DATA / "vocab.txt",
)

print(
    "Controlled subset: "
    f"dev={len(dev_rows)}, "
    f"test={len(test_rows)}, "
    f"joint inference={len(joint_rows)}"
)


# =============================================================================
# Run NSM+h and ReaRev inference
# =============================================================================

NSM_RAW = (
    NSM_CKPT_DIR / "nsm_h_raw_test.info"
)

REAREV_RAW = (
    REAREV_CKPT_DIR / "rearev_raw_test.info"
)

inference_start_time = time.time()


# -----------------------------------------------------------------------------
# NSM+h
# -----------------------------------------------------------------------------

if valid_raw_file(NSM_RAW):
    print(
        "\nValid NSM+h output already exists. "
        "Skipping NSM+h inference."
    )

else:
    if NSM_RAW.exists():
        NSM_RAW.unlink()

    run_logged(
        command=[
            sys.executable,
            "-u",
            "main_nsm.py",
            "--model_name",
            "gnn",
            "--data_folder",
            str(COMPAT_DATA) + "/",
            "--checkpoint_dir",
            str(NSM_CKPT_DIR) + "/",
            "--load_experiment",
            NSM_CKPT.name,
            "--experiment_name",
            "nsm_h_raw",
            "--test_batch_size",
            "20",
            "--num_step",
            "3",
            "--entity_dim",
            "50",
            "--word_dim",
            "300",
            "--kg_dim",
            "100",
            "--kge_dim",
            "100",
            "--encode_type",
            "--use_self_loop",
            "--reason_kb",
            "--loss_type",
            "kl",
            "--eps",
            "1.000001",
            "--is_eval",
        ],
        cwd=NSM_REPO,
        log_path=OUT_DIR / "nsm_execution_log_v1.txt",
    )

if not valid_raw_file(NSM_RAW):
    raise RuntimeError(
        f"NSM+h output is missing or incomplete:\n{NSM_RAW}"
    )

print("NSM+h inference output verified: 854 rows.")

torch.cuda.empty_cache()


# -----------------------------------------------------------------------------
# ReaRev
# -----------------------------------------------------------------------------

if valid_raw_file(REAREV_RAW):
    print(
        "\nValid ReaRev output already exists. "
        "Skipping ReaRev inference."
    )

else:
    if REAREV_RAW.exists():
        print("Removing incomplete ReaRev output:", REAREV_RAW)
        REAREV_RAW.unlink()

    rearev_environment = {
        "TOKENIZERS_PARALLELISM": "false",
        "HF_HUB_DISABLE_XET": "1",
        "HF_HUB_ENABLE_HF_TRANSFER": "0",
        "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    }

    run_logged(
        command=[
            sys.executable,
            "-u",
            "main.py",
            "ReaRev",
            "--entity_dim",
            "50",
            "--data_folder",
            str(COMPAT_DATA) + "/",
            "--lm",
            "sbert",
            "--num_iter",
            "3",
            "--num_ins",
            "2",
            "--num_gnn",
            "3",
            "--relation_word_emb",
            "True",
            "--checkpoint_dir",
            str(REAREV_CKPT_DIR) + "/",
            "--load_experiment",
            REAREV_CKPT.name,
            "--experiment_name",
            "rearev_raw",
            "--test_batch_size",
            "4",
            "--eps",
            "1.000001",
            "--is_eval",
            "--name",
            "webqsp",
        ],
        cwd=REAREV_REPO,
        log_path=REAREV_LOG,
        env=rearev_environment,
    )

if not valid_raw_file(REAREV_RAW):
    raise RuntimeError(
        f"ReaRev output is missing or incomplete:\n{REAREV_RAW}\n\n"
        f"Inspect the complete execution log:\n{REAREV_LOG}"
    )

print("ReaRev inference output verified: 854 rows.")

inference_minutes = (
    time.time() - inference_start_time
) / 60


# =============================================================================
# Parse raw prediction files
# =============================================================================

def load_raw_predictions(path, model_name):

    raw_predictions = [
        json.loads(line)
        for line in open(
            path,
            encoding="utf-8",
        )
        if line.strip()
    ]

    if len(raw_predictions) != len(joint_rows):
        raise AssertionError(
            f"{model_name}: expected {len(joint_rows)} predictions, "
            f"found {len(raw_predictions)}"
        )

    output = []

    for question_row, prediction in zip(
        joint_rows,
        raw_predictions,
    ):
        candidates = []

        for entity, probability in prediction.get("cand", []):

            if model_name == "NSM+h":
                entity = entity_ids[int(entity)]

            else:
                entity = str(entity)

            candidates.append(
                (
                    entity,
                    float(probability),
                )
            )

        candidates.sort(
            key=lambda item: item[1],
            reverse=True,
        )

        question_id = str(question_row["id"])

        output.append({
            "qid": question_id,
            "split": (
                "dev"
                if question_id in dev_qids
                else "test"
            ),
            "gold": sorted(
                gold_entity_ids(question_row)
            ),
            "candidates": candidates,
        })

    return output


nsm_raw_predictions = load_raw_predictions(
    NSM_RAW,
    "NSM+h",
)

rearev_raw_predictions = load_raw_predictions(
    REAREV_RAW,
    "ReaRev",
)


# =============================================================================
# Reconstruct answer sets
# =============================================================================

def local_entity_count(row):

    indices = set()

    for value in row["entities"]:
        index = graph_entity_index(value)

        if index is not None:
            indices.add(index)

    for value in row["subgraph"]["entities"]:
        index = graph_entity_index(value)

        if index is not None:
            indices.add(index)

    return len(indices)


MAX_LOCAL_ENTITY = max(
    local_entity_count(row)
    for row in joint_rows
)

print(
    "Maximum local entities in controlled joint split:",
    MAX_LOCAL_ENTITY,
)


def answer_set_from_mass(
    candidates,
    epsilon,
    max_local_entity=MAX_LOCAL_ENTITY,
):
    probability_floor = (
        1.0 - epsilon
    ) / max_local_entity

    retained_candidates = [
        (entity, probability)
        for entity, probability in candidates
        if probability >= probability_floor
    ]

    predicted_answers = []
    cumulative_probability = 0.0

    for entity, probability in retained_candidates:

        predicted_answers.append(entity)
        cumulative_probability += probability

        if cumulative_probability > epsilon:
            break

    return set(predicted_answers)


def per_question_records(
    raw_rows,
    epsilon,
    model_name,
):
    records = []

    for item in raw_rows:

        gold_answers = set(item["gold"])

        predicted_answers = answer_set_from_mass(
            item["candidates"],
            epsilon,
        )

        true_positives = len(
            gold_answers & predicted_answers
        )

        false_positives = len(
            predicted_answers - gold_answers
        )

        false_negatives = len(
            gold_answers - predicted_answers
        )

        precision = (
            true_positives / len(predicted_answers)
            if predicted_answers
            else 0.0
        )

        recall = (
            true_positives / len(gold_answers)
            if gold_answers
            else 1.0
        )

        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        )

        top_prediction = (
            item["candidates"][0][0]
            if item["candidates"]
            else None
        )

        records.append({
            "model": model_name,
            "qid": item["qid"],
            "split": item["split"],
            "epsilon": float(epsilon),
            "gold_answers": sorted(gold_answers),
            "predicted_answers": sorted(predicted_answers),
            "true_positives": true_positives,
            "false_positives": false_positives,
            "false_negatives": false_negatives,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "exact_match": float(
                predicted_answers == gold_answers
            ),
            "complete_recall": float(
                gold_answers.issubset(predicted_answers)
            ),
            "hits_at_1": (
                float(top_prediction in gold_answers)
                if top_prediction is not None
                else 0.0
            ),
            "predicted_answer_count": len(
                predicted_answers
            ),
            "gold_answer_count": len(
                gold_answers
            ),
        })

    return records


def aggregate(records):

    if not records:
        raise ValueError("No evaluation records provided.")

    total_true_positives = sum(
        record["true_positives"]
        for record in records
    )

    total_false_positives = sum(
        record["false_positives"]
        for record in records
    )

    total_false_negatives = sum(
        record["false_negatives"]
        for record in records
    )

    micro_precision = (
        total_true_positives
        / (
            total_true_positives
            + total_false_positives
        )
        if (
            total_true_positives
            + total_false_positives
        )
        else 0.0
    )

    micro_recall = (
        total_true_positives
        / (
            total_true_positives
            + total_false_negatives
        )
        if (
            total_true_positives
            + total_false_negatives
        )
        else 0.0
    )

    micro_f1 = (
        2
        * micro_precision
        * micro_recall
        / (micro_precision + micro_recall)
        if micro_precision + micro_recall
        else 0.0
    )

    return {
        "questions": len(records),
        "macro_precision": float(
            np.mean([
                record["precision"]
                for record in records
            ])
        ),
        "macro_recall": float(
            np.mean([
                record["recall"]
                for record in records
            ])
        ),
        "macro_f1": float(
            np.mean([
                record["f1"]
                for record in records
            ])
        ),
        "exact_match": float(
            np.mean([
                record["exact_match"]
                for record in records
            ])
        ),
        "complete_recall": float(
            np.mean([
                record["complete_recall"]
                for record in records
            ])
        ),
        "hits_at_1": float(
            np.mean([
                record["hits_at_1"]
                for record in records
            ])
        ),
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "average_predicted_answers": float(
            np.mean([
                record["predicted_answer_count"]
                for record in records
            ])
        ),
        "total_true_positives": total_true_positives,
        "total_false_positives": total_false_positives,
        "total_false_negatives": total_false_negatives,
    }


def bootstrap_confidence_intervals(
    records,
    samples=2000,
    seed=20260922,
):
    random_generator = np.random.default_rng(seed)

    number_of_records = len(records)

    f1_values = np.asarray(
        [
            record["f1"]
            for record in records
        ],
        dtype=float,
    )

    exact_values = np.asarray(
        [
            record["exact_match"]
            for record in records
        ],
        dtype=float,
    )

    complete_values = np.asarray(
        [
            record["complete_recall"]
            for record in records
        ],
        dtype=float,
    )

    bootstrapped_f1 = []
    bootstrapped_exact = []
    bootstrapped_complete = []

    for _ in range(samples):

        indices = random_generator.integers(
            0,
            number_of_records,
            number_of_records,
        )

        bootstrapped_f1.append(
            float(f1_values[indices].mean())
        )

        bootstrapped_exact.append(
            float(exact_values[indices].mean())
        )

        bootstrapped_complete.append(
            float(complete_values[indices].mean())
        )

    def interval(values):
        return [
            float(np.percentile(values, 2.5)),
            float(np.percentile(values, 97.5)),
        ]

    return {
        "bootstrap_samples": samples,
        "macro_f1_ci95": interval(
            bootstrapped_f1
        ),
        "exact_match_ci95": interval(
            bootstrapped_exact
        ),
        "complete_recall_ci95": interval(
            bootstrapped_complete
        ),
    }


# =============================================================================
# Development-only threshold selection
# =============================================================================

EPSILON_GRID = [
    round(value, 2)
    for value in np.arange(
        0.05,
        1.00,
        0.01,
    )
]

threshold_sweep = []
selected_thresholds = {}

external_models = [
    (
        "NSM+h",
        nsm_raw_predictions,
    ),
    (
        "ReaRev",
        rearev_raw_predictions,
    ),
]

for model_name, raw_predictions in external_models:

    development_predictions = [
        row
        for row in raw_predictions
        if row["split"] == "dev"
    ]

    if len(development_predictions) != 118:
        raise AssertionError(
            f"{model_name}: expected 118 development predictions."
        )

    model_sweep = []

    for epsilon in EPSILON_GRID:

        development_records = per_question_records(
            development_predictions,
            epsilon,
            model_name,
        )

        metrics = aggregate(
            development_records
        )

        sweep_result = {
            "model": model_name,
            "epsilon": epsilon,
            **metrics,
        }

        threshold_sweep.append(
            sweep_result
        )

        model_sweep.append(
            sweep_result
        )

    # Selection criterion:
    # 1. highest development macro F1;
    # 2. highest exact match;
    # 3. fewer predicted answers;
    # 4. lower epsilon.
    selected_thresholds[model_name] = max(
        model_sweep,
        key=lambda row: (
            row["macro_f1"],
            row["exact_match"],
            -row["average_predicted_answers"],
            -row["epsilon"],
        ),
    )


# =============================================================================
# Untouched test evaluation
# =============================================================================

all_test_records = []
external_summary = {}

for model_name, raw_predictions in external_models:

    selected_development_result = (
        selected_thresholds[model_name]
    )

    selected_epsilon = (
        selected_development_result["epsilon"]
    )

    test_predictions = [
        row
        for row in raw_predictions
        if row["split"] == "test"
    ]

    if len(test_predictions) != 736:
        raise AssertionError(
            f"{model_name}: expected 736 test predictions."
        )

    test_records = per_question_records(
        test_predictions,
        selected_epsilon,
        model_name,
    )

    all_test_records.extend(test_records)

    selected_test_metrics = aggregate(
        test_records
    )

    author_default_records = per_question_records(
        test_predictions,
        epsilon=0.95,
        model_name=model_name,
    )

    author_default_metrics = aggregate(
        author_default_records
    )

    checkpoint_path = (
        NSM_CKPT
        if model_name == "NSM+h"
        else REAREV_CKPT
    )

    external_summary[model_name] = {
        "checkpoint": str(checkpoint_path),
        "threshold_selection_split": "development",
        "selected_epsilon": selected_epsilon,
        "development_metrics_at_selected_epsilon": {
            key: selected_development_result[key]
            for key in [
                "questions",
                "macro_precision",
                "macro_recall",
                "macro_f1",
                "exact_match",
                "complete_recall",
                "hits_at_1",
                "micro_f1",
                "average_predicted_answers",
            ]
        },
        "test_metrics_at_selected_epsilon": (
            selected_test_metrics
        ),
        "test_bootstrap_ci": (
            bootstrap_confidence_intervals(
                test_records
            )
        ),
        "test_metrics_at_author_default_epsilon_0.95": (
            author_default_metrics
        ),
    }


# =============================================================================
# Save outputs
# =============================================================================

SUMMARY_PATH = (
    OUT_DIR / "external_baseline_summary_v1.json"
)

TABLE_PATH = (
    OUT_DIR / "external_baseline_results_table_v1.csv"
)

PER_QUESTION_PATH = (
    OUT_DIR / "external_baseline_per_question_v1.jsonl"
)

SWEEP_PATH = (
    OUT_DIR / "external_baseline_threshold_sweep_v1.csv"
)

MANIFEST_PATH = (
    OUT_DIR / "external_baseline_run_manifest_v1.json"
)

INVENTORY_PATH = (
    OUT_DIR
    / "artifact_inventory_external_baselines_v1.json"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        external_summary,
        handle,
        indent=2,
    )

with open(
    PER_QUESTION_PATH,
    "w",
    encoding="utf-8",
) as handle:
    for record in all_test_records:
        handle.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

pd.DataFrame(
    threshold_sweep
).to_csv(
    SWEEP_PATH,
    index=False,
)


# =============================================================================
# Merge SetStop and external baseline results
# =============================================================================

def setstop_table_row(
    label,
    metrics,
    comparison_type,
):
    return {
        "Method": label,
        "Comparison Type": comparison_type,
        "Macro Precision": metrics["precision"]["mean"],
        "Macro Recall": metrics["recall"]["mean"],
        "Macro F1": metrics["answer_f1"]["mean"],
        "Exact Match": metrics["exact_match"]["mean"],
        "Complete Recall": metrics["complete_recall"]["mean"],
        "Micro F1": metrics["micro_f1"],
        "Hits@1": "",
        "Actions": metrics["actions"]["mean"],
        "Retrieved Edges": (
            metrics["retrieved_edges"]["mean"]
        ),
        "Answer Cutoff": (
            "SetStop answer threshold 0.67"
        ),
    }


table_rows = []

if SETSTOP_SUMMARY.exists():

    with open(
        SETSTOP_SUMMARY,
        encoding="utf-8",
    ) as handle:
        setstop_results = json.load(handle)

    internal_methods = [
        (
            "relation_fixed_e2e",
            "Relation heuristic, fixed budget",
        ),
        (
            "model_fixed_e2e",
            "Learned ranker, fixed budget",
        ),
        (
            "setstop_e2e",
            "SetStop",
        ),
        (
            "oracle_rank_e2e",
            "Oracle ranker",
        ),
    ]

    for result_key, display_name in internal_methods:

        if result_key not in setstop_results:
            print(
                "Warning: missing SetStop result:",
                result_key,
            )
            continue

        table_rows.append(
            setstop_table_row(
                label=display_name,
                metrics=setstop_results[result_key],
                comparison_type="controlled internal",
            )
        )

else:
    print(
        "Warning: SetStop result file was not found:",
        SETSTOP_SUMMARY,
    )


for model_name in [
    "NSM+h",
    "ReaRev",
]:
    result = external_summary[model_name]

    test_metrics = (
        result["test_metrics_at_selected_epsilon"]
    )

    table_rows.append({
        "Method": model_name,
        "Comparison Type": (
            "controlled external checkpoint"
        ),
        "Macro Precision": (
            test_metrics["macro_precision"]
        ),
        "Macro Recall": (
            test_metrics["macro_recall"]
        ),
        "Macro F1": (
            test_metrics["macro_f1"]
        ),
        "Exact Match": (
            test_metrics["exact_match"]
        ),
        "Complete Recall": (
            test_metrics["complete_recall"]
        ),
        "Micro F1": (
            test_metrics["micro_f1"]
        ),
        "Hits@1": (
            test_metrics["hits_at_1"]
        ),
        "Actions": "N/A",
        "Retrieved Edges": "N/A",
        "Answer Cutoff": (
            "development-selected epsilon="
            f"{result['selected_epsilon']:.2f}"
        ),
    })

results_table = pd.DataFrame(table_rows)

results_table.to_csv(
    TABLE_PATH,
    index=False,
)


# =============================================================================
# Reproducibility manifest
# =============================================================================

test_qid_hash = hashlib.sha256(
    "\n".join(
        sorted(test_qids)
    ).encode("utf-8")
).hexdigest()

manifest = {
    "notebook": (
        "07_SetStop_External_Baselines_"
        "NSM_ReaRev.ipynb"
    ),
    "evaluation_only": True,
    "runtime": "GPU",
    "gpu": gpu_name,
    "gpu_memory_gb": gpu_memory_gb,
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "transformers_before_compatibility_install": (
        transformers_before
    ),
    "rearev_subprocess_transformers_version": (
        rearev_transformers_version
    ),
    "data_directory": str(DATA_DIR),
    "subset_definition": (
        "2-50 mapped gold answers and every mapped "
        "gold answer present in the local graph"
    ),
    "development_questions": len(dev_rows),
    "test_questions": len(test_rows),
    "joint_inference_questions": len(joint_rows),
    "test_qid_sha256": test_qid_hash,
    "threshold_rule": (
        "maximize development macro F1; "
        "tie-break using exact match, fewer predictions, "
        "and lower epsilon"
    ),
    "threshold_grid": (
        "0.05 to 0.99 inclusive, step 0.01"
    ),
    "test_used_for_threshold_selection": False,
    "repositories": {
        "NSM+h": {
            "url": (
                "https://github.com/"
                "RichardHGL/WSDM2021_NSM"
            ),
            "commit": NSM_COMMIT,
            "checkpoint_sha256": (
                sha256_file(NSM_CKPT)
            ),
        },
        "ReaRev": {
            "url": (
                "https://github.com/"
                "cmavro/ReaRev_KGQA"
            ),
            "commit": REAREV_COMMIT,
            "checkpoint_sha256": (
                sha256_file(REAREV_CKPT)
            ),
        },
    },
    "inference_minutes_this_run": (
        inference_minutes
    ),
    "notes": [
        (
            "Both external checkpoints are evaluated "
            "on the same 736 question IDs as SetStop."
        ),
        (
            "Both use the same NSM-preprocessed "
            "question-specific WebQSP graphs."
        ),
        (
            "Actions and retrieved edges are N/A for "
            "NSM+h and ReaRev because they use full-graph "
            "message passing rather than explicit "
            "SetStop frontier actions."
        ),
        (
            "The main table uses an answer-set epsilon "
            "selected only on the 118-question "
            "development subset."
        ),
        (
            "Test results were not used for threshold "
            "selection."
        ),
    ],
}

with open(
    MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )


# =============================================================================
# Artifact inventory
# =============================================================================

artifact_paths = [
    SUMMARY_PATH,
    TABLE_PATH,
    PER_QUESTION_PATH,
    SWEEP_PATH,
    MANIFEST_PATH,
]

nsm_log = OUT_DIR / "nsm_execution_log_v1.txt"

if nsm_log.exists():
    artifact_paths.append(nsm_log)

if REAREV_LOG.exists():
    artifact_paths.append(REAREV_LOG)

inventory = {}

for path in artifact_paths:
    inventory[path.name] = {
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }

with open(
    INVENTORY_PATH,
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        inventory,
        handle,
        indent=2,
    )


# =============================================================================
# Final results
# =============================================================================

print("\n" + "=" * 100)
print("CONTROLLED EXTERNAL-BASELINE RESULTS")
print("=" * 100)

display(results_table)

for model_name in [
    "NSM+h",
    "ReaRev",
]:
    result = external_summary[model_name]

    selected_epsilon = (
        result["selected_epsilon"]
    )

    metrics = (
        result["test_metrics_at_selected_epsilon"]
    )

    confidence_interval = (
        result["test_bootstrap_ci"]["macro_f1_ci95"]
    )

    print(
        f"\n{model_name}\n"
        f"  Selected development epsilon: "
        f"{selected_epsilon:.2f}\n"
        f"  Test Macro F1: "
        f"{metrics['macro_f1']:.4f}\n"
        f"  Macro F1 95% CI: "
        f"[{confidence_interval[0]:.4f}, "
        f"{confidence_interval[1]:.4f}]\n"
        f"  Exact Match: "
        f"{metrics['exact_match']:.4f}\n"
        f"  Complete Recall: "
        f"{metrics['complete_recall']:.4f}\n"
        f"  Hits@1: "
        f"{metrics['hits_at_1']:.4f}\n"
        f"  Micro F1: "
        f"{metrics['micro_f1']:.4f}"
    )

print("\nSaved output files:")

for path in artifact_paths + [INVENTORY_PATH]:
    print(path)

print("\n" + "=" * 100)
print("EXTERNAL BASELINE EVALUATION COMPLETED")
print("=" * 100)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SETSTOP-KGQA CONTROLLED EXTERNAL BASELINES: NSM+h AND ReaRev

RUN: /usr/bin/python3 -m pip install -q --upgrade gdown>=5.2.0 pandas>=2.0 tqdm>=4.60
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you h

,Method,Comparison Type,Macro Precision,Macro Recall,Macro F1,Exact Match,Complete Recall,Micro F1,Hits@1,Actions,Retrieved Edges,Answer Cutoff
0,"Relation heuristic, fixed budget",controlled internal,0.268308,0.446065,0.301682,0.112772,0.406250,0.386602,,12.0,148.555707,SetStop answer threshold 0.67
1,"Learned ranker, fixed budget",controlled internal,0.304683,0.496577,0.338372,0.127717,0.459239,0.414755,,12.0,171.036685,SetStop answer threshold 0.67
2,SetStop,controlled internal,0.382553,0.435909,0.385640,0.277174,0.392663,0.472246,,5.952446,110.275815,SetStop answer threshold 0.67
3,Oracle ranker,controlled internal,0.464274,0.528016,0.469762,0.338315,0.475543,0.598098,,8.097826,166.38587,SetStop answer threshold 0.67
4,NSM+h,controlled external checkpoint,0.751798,0.824050,0.756977,0.508152,0.688859,0.639364,0.816576,N/A,N/A,development-selected epsilon=0.96
5,ReaRev,controlled external checkpoint,0.812919,0.857257,0.812286,0.505435,0.652174,0.765007,0.84375,N/A,N/A,development-selected epsilon=0.89



NSM+h
  Selected development epsilon: 0.96
  Test Macro F1: 0.7570
  Macro F1 95% CI: [0.7320, 0.7819]
  Exact Match: 0.5082
  Complete Recall: 0.6889
  Hits@1: 0.8166
  Micro F1: 0.6394

ReaRev
  Selected development epsilon: 0.89
  Test Macro F1: 0.8123
  Macro F1 95% CI: [0.7896, 0.8346]
  Exact Match: 0.5054
  Complete Recall: 0.6522
  Hits@1: 0.8438
  Micro F1: 0.7650

Saved output files:
/content/drive/MyDrive/SetStop_KGQA/outputs/external_baselines_v1/external_baseline_summary_v1.json
/content/drive/MyDrive/SetStop_KGQA/outputs/external_baselines_v1/external_baseline_results_table_v1.csv
/content/drive/MyDrive/SetStop_KGQA/outputs/external_baselines_v1/external_baseline_per_question_v1.jsonl
/content/drive/MyDrive/SetStop_KGQA/outputs/external_baselines_v1/external_baseline_threshold_sweep_v1.csv
/content/drive/MyDrive/SetStop_KGQA/outputs/external_baselines_v1/external_baseline_run_manifest_v1.json
/content/drive/MyDrive/SetStop_KGQA/outputs/external_baselines_v1/rearev_execut